# EasyOCR ? Local OPRB evaluation
Open in the Lab_1 directory with the project's **evnv** kernel and run all cells. This notebook evaluates all 9,000 occluded test pages on one local CUDA GPU. Each prediction is committed to results/ before processing the next page. Re-run after interruption to retry unfinished pages. The same ten seed-42 pilot pages run first. The final tables require all 9,000 successful results.
One document per inference call; recognition batches up to 16 detected regions within that page.

In [1]:
from pathlib import Path
import sys
if Path(sys.prefix).resolve() != Path("evnv").resolve():
    raise RuntimeError("Select Lab_1/evnv/Scripts/python.exe as the notebook kernel.")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("The project kernel needs CUDA-enabled PyTorch.")
torch.set_num_threads(2)
CACHE = Path.home() / ".cache" / "oprb" / "data"
OUTPUT = Path("results")
OUTPUT.mkdir(exist_ok=True)
print(torch.cuda.get_device_name(), torch.__version__)

NVIDIA GeForce RTX 5060 Laptop GPU 2.14.1+cu130


In [2]:
import hashlib, json, random, shutil, sqlite3, tarfile, time, unicodedata
from collections import Counter
from importlib.metadata import version
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image
from jiwer import cer, wer
import pandas as pd

REPO = "kpurkayastha/OPRB"
DATA_REV = 'c7ab9fdf2bf33f297bba62e4842d3e5094d11ec2'
ROOT = CACHE / DATA_REV
COUNTS = {"text": 6000, "nontext": 3000}
pages = []
for region, count in COUNTS.items():
    for kind, suffix in [("img", ".jpg"), ("gt", ".txt")]:
        target = ROOT / region / kind
        target.mkdir(parents=True, exist_ok=True)
        if not (target / ".complete").is_file():
            archive = hf_hub_download(REPO, f"test/{region}/{kind}.tar.gz", repo_type="dataset", revision=DATA_REV)
            seen = set()
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    name = Path(member.name).name
                    if not member.isfile() or Path(name).suffix.lower() != suffix:
                        continue
                    if name in seen:
                        raise ValueError(f"Repeated basename: {name}")
                    seen.add(name)
                    with tar.extractfile(member) as src, (target / name).open("wb") as dest:
                        shutil.copyfileobj(src, dest)
            assert len(seen) == count, (region, kind, len(seen))
            (target / ".complete").touch()
    imgs = {p.stem for p in (ROOT / region / "img").glob("*.jpg")}
    refs = {p.stem for p in (ROOT / region / "gt").glob("*.txt")}
    assert imgs == refs and len(imgs) == count, (region, len(imgs), len(refs))
    pages.extend((region, stem) for stem in sorted(imgs))
pilot = [key for region in COUNTS for key in
         random.Random(42).sample([p for p in pages if p[0] == region], 5)]
pilot_keys = set(pilot)
selected = pilot + [p for p in pages if p not in pilot_keys]
print(Counter(r for r, _ in pages))

D:\APCS\Y3\7_CS418_Introduction_to_NLP\Lab_1\evnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Counter({'text': 6000, 'nontext': 3000})


In [3]:
def reference(region, stem):
    tokens = []
    path = ROOT / region / "gt" / (stem + ".txt")
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        fields = line.split("\t")
        if len(fields) not in (10, 11):
            raise ValueError(f"Malformed GT row in {path.name}: {line[:60]!r}")
        if fields[0].strip():
            tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f"Empty GT: {path.name}")
    return " ".join(tokens)

def normalize(s):
    return " ".join(unicodedata.normalize("NFKC", s).casefold().split())

def metrics(ref, hyp):
    ref, hyp = normalize(ref), normalize(hyp)
    a, b = Counter(ref.split()), Counter(hyp.split())
    hits = sum((a & b).values())
    p = hits / sum(b.values()) if b else 0
    r = hits / sum(a.values())
    return cer(ref, hyp), wer(ref, hyp), p, r, 2*p*r/(p+r) if hits else 0

assert metrics("Ａ b", "a B") == (0, 0, 1, 1, 1)
assert metrics("a", "") == (1, 1, 0, 0, 0)
assert metrics("a a", "a")[2:] == (1, .5, 2/3)
assert metrics("a b", "b a")[1:] == (1, 1, 1, 1)

In [4]:
from test_easyocr_local import prepare_easyocr
import easyocr
from contextlib import closing
from tqdm.auto import tqdm

folder = Path.home() / ".cache/oprb/easyocr-en"
prepare_easyocr(folder)
reader = easyocr.Reader(["en"], gpu=True, model_storage_directory=str(folder), download_enabled=False)
files = [folder / name for name in ("craft_mlt_25k.pth", "english_g2.pth")]
config = dict(model="easyocr", dataset=DATA_REV, device=torch.cuda.get_device_name(), workers=1,
              versions={p: version(p) for p in ("torch","torchvision","easyocr","opencv-python-headless","numpy","pillow")},
              cuda=torch.version.cuda, threads=torch.get_num_threads(), languages=["en"],
              detector="craft", recognizer="english_g2", decoder="greedy", batch_size=16,
              weights_mib=sum(p.stat().st_size for p in files)/2**20,
              revisions={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in files})
cfg = json.dumps(config, sort_keys=True)
checkpoint = OUTPUT / ("easyocr-" + hashlib.sha256(cfg.encode()).hexdigest()[:16] + "-0.sqlite")
def transcribe(region, stem):
    path = ROOT / region / "img" / (stem + ".jpg")
    return " ".join(reader.readtext(str(path), detail=0, batch_size=16, decoder="greedy"))

with closing(sqlite3.connect(checkpoint)) as db:
    db.execute("CREATE TABLE IF NOT EXISTS config (json TEXT PRIMARY KEY)")
    db.execute("INSERT OR IGNORE INTO config VALUES (?)", (cfg,))
    db.execute("""CREATE TABLE IF NOT EXISTS pages (region TEXT, stem TEXT, prediction TEXT,
               seconds REAL, truncated INTEGER, error TEXT, PRIMARY KEY(region, stem))""")
    db.commit()
    done = set(db.execute("SELECT region, stem FROM pages WHERE error IS NULL"))
    pending = [key for key in selected if key not in done]
    if pending:
        transcribe(*pending[0])  # Warm-up outside the timed loop
    for region, stem in tqdm(pending, desc="EasyOCR"):
        try:
            torch.cuda.synchronize()
            start = time.perf_counter()
            pred = transcribe(region, stem)
            torch.cuda.synchronize()
            row = (region, stem, pred, time.perf_counter()-start, 0, None)
        except Exception as exc:
            row = (region, stem, None, None, None, repr(exc))
        with db:
            db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?,?,?,?)", row)
        if row[-1]:
            raise RuntimeError(f"{region}/{stem}: {row[-1]}")
print(checkpoint)

Verified cached craft_mlt_25k.pth


Verified cached english_g2.pth


EasyOCR:   0%|          | 0/9000 [00:00<?, ?it/s]

EasyOCR:   0%|          | 1/9000 [00:01<3:38:41,  1.46s/it]

EasyOCR:   0%|          | 2/9000 [00:02<2:31:28,  1.01s/it]

EasyOCR:   0%|          | 3/9000 [00:02<1:56:30,  1.29it/s]

... (8,989 outputs left out)

EasyOCR: 100%|█████████▉| 8999/9000 [2:08:28<00:00,  1.49it/s]

EasyOCR: 100%|██████████| 9000/9000 [2:08:29<00:00,  1.56it/s]

EasyOCR: 100%|██████████| 9000/9000 [2:08:29<00:00,  1.17it/s]

results\easyocr-a1543d354a5eca0b-0.sqlite


In [5]:
from contextlib import closing
def occlusion_metadata(stem):
    import re
    parts = stem.split("__", 2)
    density = parts[1] if len(parts) > 1 and re.fullmatch(r"[0-9]+(?:p[0-9]+)?pct", parts[1]) else "unknown"
    return parts[0], density

def score_results():
    rows = []
    refs = {}
    for path in [checkpoint]:
        with closing(sqlite3.connect(path)) as db:
            cfg, = db.execute("SELECT json FROM config").fetchone()
            config = json.loads(cfg)
            if config["dataset"] != DATA_REV:
                continue
            for region, stem, pred, seconds, trunc, error in db.execute("SELECT * FROM pages"):
                key = (region, stem)
                if key not in refs:
                    refs[key] = reference(*key)
                scores = (None,)*5 if error else metrics(refs[key], pred)
                parts = occlusion_metadata(stem)
                rows.append((config["model"], hashlib.sha256(cfg.encode()).hexdigest()[:12], config["device"],
                             config["workers"], config["weights_mib"], region, stem, parts[0], parts[1],
                             pred, refs[key], seconds, trunc, error, *scores))
    cols = ["model","config","device","workers","weights_mib","region","stem","type","density",
            "prediction","gt","seconds","truncated","error","cer","wer","precision","recall","f1"]
    return pd.DataFrame(rows, columns=cols)

results = score_results()
assert len(results) and not results.duplicated(["model","config","region","stem"]).any()
assert set(zip(results.region, results.stem)) == set(pages) and results.error.isna().all()
keys = ["model","config"]
values = ["cer","wer","precision","recall","f1","seconds"]
coverage = results.groupby(keys + ["region"]).agg(attempted=("stem","size"), ok=("seconds","count"),
                                                   truncated=("truncated","sum"))
coverage["expected"] = coverage.index.get_level_values("region").map(COUNTS)
display(coverage)
ok = results[results.error.isna()]
display(ok.groupby(keys+["device","weights_mib"])[values].mean())
for groups in (["region"], ["region","type"], ["region","density"]):
    display(ok.groupby(keys+groups)[values].mean().join(ok.groupby(keys+groups).size().rename("pages")))
latency = ok.groupby(keys+["workers","region"]).seconds.mean().unstack().reindex(columns=list(COUNTS))
display((latency.mul(pd.Series(COUNTS)).sum(axis=1, skipna=False) /
         (3600 * latency.index.get_level_values("workers"))).rename("projected_wall_hours"))
display(pd.DataFrame([(r, *occlusion_metadata(s)) for r,s in pages],
                     columns=["region","type","density"]).value_counts().rename("dataset_pages"))
# Locate qualitative candidates. Inspect the same image crop, GT and predictions before using them.
for region, group in ok.groupby("region"):
    print(region)
    display(pd.concat([group.nsmallest(1,"wer"),group.nlargest(1,"wer")])[
        ["model","stem","wer","gt","prediction"]])
# Example: display(Image.open(ROOT / region / "img" / (stem + ".jpg")))

attempted    ok  truncated  expected
model   config       region                                       
easyocr a1543d354a5e nontext       3000  3000          0      3000
                     text          6000  6000          0      6000

,,,,cer,wer,precision,recall,f1,seconds
model,config,device,weights_mib,,,,,,
easyocr,a1543d354a5e,NVIDIA GeForce RTX 5060 Laptop GPU,93.742682,0.599304,0.827841,0.438559,0.301553,0.350143,0.843172


cer       wer  precision    recall  \
model   config       region                                             
easyocr a1543d354a5e nontext  0.668486  0.858769   0.424500  0.273719   
                     text     0.564713  0.812376   0.445588  0.315469   

                                    f1   seconds  pages  
model   config       region                              
easyocr a1543d354a5e nontext  0.321551  0.855107   3000  
                     text     0.364438  0.837204   6000

cer       wer  precision  \
model   config       region  type                                            
easyocr a1543d354a5e nontext Black_Ink       0.651237  0.843151   0.431656   
                             Sim             0.662795  0.856746   0.409886   
                             Through_Dust    0.683034  0.876442   0.440349   
                             Through_Stamp   0.679609  0.863568   0.408836   
                             White_Burnt     0.667282  0.855631   0.427686   
                             White_Whitener  0.667990  0.857614   0.422331   
                     text    Black_Ink       0.562761  0.811871   0.452107   
                             Sim             0.561084  0.809393   0.452584   
                             Through_Dust    0.557134  0.808398   0.440853   
                             Through_Stamp   0.578171  0.819692   0.442311   
                             White_Burnt     0.558982  0.808270   0.446351   
                             White_Whitener  0.570148  0.816634   0.439323   

                                               recall        f1   seconds  \
model   config       region  type                                           
easyocr a1543d354a5e nontext Black_Ink       0.278955  0.328004  0.833375   
                             Sim             0.268006  0.314478  0.886633   
                             Through_Dust    0.293747  0.340182  0.849410   
                             Through_Stamp   0.251211  0.300017  0.839332   
                             White_Burnt     0.275533  0.323204  0.882670   
                             White_Whitener  0.269126  0.317592  0.842740   
                     text    Black_Ink       0.320685  0.370520  0.834992   
                             Sim             0.324136  0.373319  0.864968   
                             Through_Dust    0.318655  0.364995  0.850213   
                             Through_Stamp   0.302010  0.353299  0.803263   
                             White_Burnt     0.316957  0.365838  0.832029   
                             White_Whitener  0.310372  0.358658  0.837760   

                                             pages  
model   config       region  type                   
easyocr a1543d354a5e nontext Black_Ink         537  
                             Sim               423  
                             Through_Dust      537  
                             Through_Stamp     429  
                             White_Burnt       537  
                             White_Whitener    537  
                     text    Black_Ink        1000  
                             Sim              1000  
                             Through_Dust     1000  
                             Through_Stamp    1000  
                             White_Burnt      1000  
                             White_Whitener   1000

cer       wer  precision    recall  \
model   config       region  density                                            
easyocr a1543d354a5e nontext 0p5pct   0.658550  0.850813   0.427834  0.279677   
                             1p0pct   0.674354  0.868736   0.437565  0.285985   
                             2p0pct   0.668520  0.854181   0.425597  0.271968   
                             unknown  0.671261  0.860181   0.409357  0.259549   
                     text    0p5pct   0.562352  0.810436   0.448790  0.321256   
                             1p0pct   0.558842  0.809950   0.443265  0.317944   
                             2p0pct   0.565591  0.813445   0.442179  0.311082   
                             unknown  0.569627  0.814542   0.447447  0.313073   

                                            f1   seconds  pages  
model   config       region  density                             
easyocr a1543d354a5e nontext 0p5pct   0.326114  0.849375    685  
                             1p0pct   0.334929  0.869819    753  
                             2p0pct   0.320188  0.835781    710  
                             unknown  0.307196  0.862816    852  
                     text    0p5pct   0.369615  0.857998   1276  
                             1p0pct   0.365454  0.837077   1364  
                             2p0pct   0.360223  0.822364   1360  
                             unknown  0.363309  0.834115   2000

model    config        workers
easyocr  a1543d354a5e  1          2.107929
Name: projected_wall_hours, dtype: float64

region   type            density
text     Sim             unknown    1000
         Through_Stamp   unknown    1000
nontext  Through_Stamp   unknown     429
         Sim             unknown     423
text     White_Burnt     2p0pct      363
         White_Whitener  1p0pct      357
         Through_Dust    2p0pct      349
                         1p0pct      345
         Black_Ink       1p0pct      338
                         0p5pct      336
                         2p0pct      326
         White_Burnt     1p0pct      324
         White_Whitener  2p0pct      322
                         0p5pct      321
         White_Burnt     0p5pct      313
         Through_Dust    0p5pct      306
nontext  Black_Ink       1p0pct      195
         White_Whitener  1p0pct      194
         Through_Dust    0p5pct      184
         White_Burnt     1p0pct      184
                         2p0pct      182
         Through_Dust    1p0pct      180
         Black_Ink       2p0pct      179
         White_Whitener 

nontext


,model,stem,wer,gt,prediction
7295,easyocr,Through_Dust__1p0pct__107.tar_1507.02879.gz_Pa...,0.259714,"6 SARFRAZ, STIEFELHAGEN: DEEP PERCEPTUAL MAPPI...",SARFRAZ STIEFELHAGEN: DEEP PERCEPTUAL MAPPING ...
7182,easyocr,Through_Dust__1p0pct__106.tar_1705.06899.gz_ML...,17.400000,Table 12: SVM - Means µ and Standard Deviation...,Table 12: SVM Means Standard and Deviations an...


text


,model,stem,wer,gt,prediction
5433,easyocr,White_Whitener__1p0pct__104.tar_1507.01565.gz_...,0.166667,8 TORSION AND GROUND STATE MAXIMA,TORSION AND GROUND STATE MAXIMA
405,easyocr,Black_Ink__1p0pct__1.tar_1801.00495.gz_sdAs1_i...,1.562130,18 I. Pelisoli et al. Table C1. Objects for wh...,"1, Pelisoli et al fit irdlicated Ta > #wex} K ..."
